# Module 4.2: Migrating Graphics-Intensive Workloads — Remote CUDA Machine Learning & NVENC Transcoding

## Homelab Expansion (Chapter 2) - Step 4: GPU Acceleration & Workload Migration

## 🎯 Learning Goals

By completing this laboratory module, you will be able to:

1. Understand the engineering rationale for **decoupling AI/GPU compute from NAS storage** (or optionally migrating the entire stack vs. Remote ML worker mode).
2. Deploy the CUDA-accelerated `immich-machine-learning:release-cuda` container on `homelab-gpu` (`10.0.0.201:3003`) with NVIDIA GPU hardware reservation.
3. Point the primary Immich Server on `homelab-pi` (`10.0.0.200:2283`) to `http://10.0.0.201:3003` over the `USW-Enterprise-8-PoE` switch and decommission the CPU-bound ML container on the Raspberry Pi.
4. Upgrade from lightweight mobile CLIP models to heavier, higher-accuracy semantic search and facial recognition models (`ViT-B-16-SigLIP` / `buffalo_l`) now backed by 6GB–8GB of GDDR6 VRAM.
5. Configure hardware-accelerated **NVIDIA NVENC** video transcoding options.

## 4.2.1 Conceptual Background: Why Offloading Graphics-Intensive Workloads Makes Sense

In Chapter 1, our 4GB Raspberry Pi 4 had to run all four Immich services (`immich-server`, `immich-postgres`, `immich-redis`, and `immich-machine-learning`) simultaneously on four 1.5GHz ARM Cortex-A72 CPU cores.

### 1. Quantitative Comparison: Raspberry Pi 4 CPU vs. 2019 ROG Zephyrus NVIDIA GPU

| Workload / Metric | Raspberry Pi 4 (4GB ARM CPU Only) | 2019 ROG Zephyrus (`homelab-gpu` NVIDIA CUDA + NVENC) | Engineering Impact |
| :--- | :--- | :--- | :--- |
| **CLIP Smart Search Inference (`1,000` Photos)** | ~35 – 50 minutes (`1` worker thread) | **~45 – 75 seconds** (`CUDAExecutionProvider` on 1,536+ CUDA cores) | **~40x speedup** during bulk album imports and re-indexing. |
| **Facial Detection & Recognition (`buffalo_l`)** | ~1.5 – 2.8 seconds per multi-person photo | **~30 – 65 milliseconds** per photo in GDDR6 VRAM | Instant face clustering as soon as photos upload from your phone. |
| **4K HEVC (H.265) Video Transcoding** | ~0.15x real-time (CPU pegged at 400%, thermal throttling) | **3x – 5x real-time** via dedicated **Turing NVENC** silicon encoder | Zero CPU bottleneck when generating web-playable video previews. |
| **Raspberry Pi RAM Saved** | `immich-machine-learning` consumes **1.2 GB – 1.8 GB RAM** on the Pi | **0 MB on the Pi** (offloaded to `homelab-gpu`'s 6GB+ VRAM & 16GB+ RAM) | Frees **~45% of the Raspberry Pi's total 4GB RAM** for PostgreSQL cache, Pi-hole DNS, and Uptime Kuma! |

### 2. Two Architectural Patterns for Workload Migration

```mermaid
flowchart LR
    subgraph Pi ["Node 1: homelab-pi (10.0.0.200) - 5W Always-On NAS"]
        Server["immich-server<br>(Port 2283 Web/API)"]
        DB["immich-postgres<br>(pgvector Metadata)"]
        Redis["immich-redis"]
        SSD["WD Blue SN5000 NVMe SSD<br>/mnt/homelab_storage"]
    end
    
    subgraph Laptop ["Node 2: homelab-gpu (10.0.0.201) - NVIDIA CUDA Worker"]
        ML["immich-machine-learning:release-cuda<br>(Port 3003 | GPU Accelerated)"]
        VRAM["NVIDIA GTX/RTX GDDR6 VRAM<br>(CLIP + InsightFace Models)"]
    end
    
    Server --> DB
    Server --> Redis
    Server --> SSD
    Server ===|"HTTP POST /predict (USW-Enterprise-8-PoE <0.3ms)"| ML
    ML --> VRAM
```

1. **Pattern A (Recommended Default): Remote CUDA Machine Learning Offload**
   - **How it works:** Keep your WD Blue SN5000 NVMe SSD, `immich-postgres`, `immich-redis`, and `immich-server` on the ultra-low-power `homelab-pi` (`10.0.0.200`), and run `immich-machine-learning:release-cuda` on `homelab-gpu` (`10.0.0.201:3003`).
   - **Why it is awesome:** Immich natively supports a list of Machine Learning URLs with automatic failover! Your photo library stays on the 5W Raspberry Pi 24/7. Even if you reboot or perform maintenance on the laptop, you can still browse and upload photos to the Pi—and as soon as the laptop is online, `immich-server` streams thumbnails over the `USW-Enterprise-8-PoE` switch to `http://10.0.0.201:3003` for blazing-fast CUDA processing.
2. **Pattern B (Optional Full Compute + NVENC Transcoding Offload):**
   - If you also shoot heavy 4K 60fps raw video and want hardware **NVENC video transcoding**, you can additionally mount `/mnt/homelab_storage/immich_library` over the switch via NFSv4 (or move the USB 3.0 NVMe enclosure directly to the laptop's 10Gbps USB 3.2 Gen 2 port) and enable `hwaccel.transcoding.yml` (`service: nvenc`). We cover both below!

## 4.2.2 Step-by-Step Laboratory Guide

### Task 4.2.1: Deploy `immich-machine-learning:release-cuda` on `homelab-gpu`
1. Connect to `homelab-gpu` (`ssh homelab-gpu`) and create the standardized homelab stack directory:
   ```bash
   mkdir -p ~/homelab/stacks/immich-ml
   cd ~/homelab/stacks/immich-ml
   ```
2. Create the CUDA-enabled `docker-compose.yml` file:
   ```bash
   cat << 'EOT' > ~/homelab/stacks/immich-ml/docker-compose.yml
   name: immich_remote_ml

   services:
     immich-machine-learning:
       container_name: immich_machine_learning
       # Important: Notice the -cuda suffix for NVIDIA GPU acceleration!
       image: ghcr.io/immich-app/immich-machine-learning:${IMMICH_VERSION:-release}-cuda
       ports:
         - "3003:3003"
       volumes:
         - model-cache:/cache
       environment:
         - TZ=America/New_York
         - MACHINE_LEARNING_CACHE_FOLDER=/cache
         # Keep models warm in GDDR6 VRAM for 10 minutes between uploads
         - MACHINE_LEARNING_MODEL_TTL=600
       deploy:
         resources:
           reservations:
             devices:
               - driver: nvidia
                 count: 1
                 capabilities:
                   - gpu
       restart: unless-stopped

   volumes:
     model-cache:
   EOT
   ```
3. Pull the `-cuda` image and launch the container on `homelab-gpu`:
   ```bash
   cd ~/homelab/stacks/immich-ml
   docker compose pull
   docker compose up -d
   ```
4. Verify that the container is healthy and listening on port `3003`:
   ```bash
   docker compose ps
   curl -s http://localhost:3003/ping && echo ""
   ```
   *(Expected output: `pong`)*

### Task 4.2.2: Switch `homelab-pi` to Use `homelab-gpu` & Retire the Pi's Local ML Container
Now let's point your primary Immich instance on `homelab-pi` to your new NVIDIA CUDA worker at `http://10.0.0.201:3003` and free up ~1.5GB of RAM on your Raspberry Pi!

1. Open your Immich Web UI in a browser (`http://10.0.0.200:2283` or `http://homelab-pi:2283`).
2. Navigate to **Administration** (top-right gear icon) $\rightarrow$ **Settings** $\rightarrow$ **Machine Learning Settings**.
3. Under **URL**, replace `http://immich-machine-learning:3003` with your ROG Zephyrus GPU node's static LAN IP across the `USW-Enterprise-8-PoE` switch:
   ```text
   http://10.0.0.201:3003
   ```
4. **Optional Model Accuracy Upgrade:** Now that you have a dedicated NVIDIA GPU with 6GB–8GB VRAM instead of a 4GB Raspberry Pi CPU:
   - Under **Smart Search**, you can keep `ViT-B-32__openai` (ultra-fast) or upgrade to **`ViT-B-16-SigLIP__webli`** for much richer natural-language photo search.
   - Click **Save** at the bottom of the settings page.
5. **Stop and Remove the Redundant CPU Machine Learning Container on `homelab-pi`:**
   SSH into your Raspberry Pi (`ssh homelab-pi`) and stop `immich_machine_learning`:
   ```bash
   cd ~/homelab/stacks/immich
   # Comment out or stop the local CPU-bound machine-learning container on the Pi
   docker compose stop immich-machine-learning
   docker compose rm -f immich-machine-learning
   free -h
   ```
   Look at `free -h` on `homelab-pi`—you just reclaimed over **1.2 GB of physical RAM** on your Raspberry Pi!

### Task 4.2.3: Trigger a Bulk GPU Job & Watch Live CUDA Execution in `nvtop`
1. On `homelab-gpu`, open two terminal panes (or run `nvtop` while tailing logs):
   ```bash
   docker logs -f immich_machine_learning
   ```
2. In your Immich Web UI (`Administration` $\rightarrow$ `Jobs`), click **Run** $\rightarrow$ **All** (or **Missing**) next to **Smart Search** and **Face Detection**.
3. Watch `docker logs -f immich_machine_learning` and `nvidia-smi` / `nvtop` on `homelab-gpu`:
   - You will see ONNX Runtime initialize **`CUDAExecutionProvider`** (instead of `CPUExecutionProvider`), load the model weights directly into your NVIDIA GPU's VRAM (`~1.1 GB VRAM`), and tear through hundreds of photos in seconds while `homelab-pi` remains completely cool and idle!

### 🖼️ Task 4.2.3 Verification: Live NVIDIA CUDA Execution in `immich_machine_learning`

**Verification Objective:** Capture a screenshot on `homelab-gpu` showing `nvidia-smi` (or `nvtop` / `docker logs immich_machine_learning`) actively executing `python /usr/src/app` on the NVIDIA GPU using `CUDAExecutionProvider`.

---

> ### 📷 User Verification Screenshot Submission Box
> *Attach your milestone verification screenshot below (drag & drop an image or use a Markdown image link):*
> 
> ```markdown
> <!-- Insert your screenshot here: -->
> 
> ```

### Task 4.2.4: Reference Guide — Enabling NVIDIA NVENC Hardware Video Transcoding
If you ever run an `immich-server` worker or media server (like Jellyfin / Plex) on `homelab-gpu` with access to your media files (either via an NFSv4 mount from `homelab-pi` or by attaching your USB NVMe SSD), you can enable hardware **Turing NVENC/NVDEC** video transcoding in two steps:

1. Download Immich's official hardware acceleration template alongside `docker-compose.yml`:
   ```bash
   wget -O hwaccel.transcoding.yml https://github.com/immich-app/immich/releases/latest/download/hwaccel.transcoding.yml
   ```
2. Inside `docker-compose.yml` under `immich-server`, uncomment the `extends` block for `nvenc`:
   ```yaml
   services:
     immich-server:
       container_name: immich_server
       image: ghcr.io/immich-app/immich-server:${IMMICH_VERSION:-release}
       extends:
         file: hwaccel.transcoding.yml
         service: nvenc
   ```
3. In the Immich Web UI under **Administration $\rightarrow$ Settings $\rightarrow$ Video Transcoding Settings $\rightarrow$ Hardware Acceleration**, select **`NVENC`** and enable **Hardware Decoding**. The Turing NVENC ASIC on your 2019 ROG Zephyrus will transcode 4K HEVC mobile videos into H.264/HEVC web streams at 300+ FPS using dedicated silicon outside the main CUDA cores!

## 🏁 Module Checkpoint & Reflection

Before proceeding to the next guide, confirm the following checklist:

- [ ] `immich-machine-learning:release-cuda` deployed and responding with `pong` on `homelab-gpu:3003`.
- [ ] Immich Machine Learning URL updated to `http://10.0.0.201:3003` in the Administration UI.
- [ ] Redundant CPU `immich-machine-learning` container stopped on `homelab-pi`, freeing ~1.2GB+ of Pi RAM.
- [ ] Live CUDA GPU acceleration verified via `nvidia-smi` / `docker logs immich_machine_learning`.
- [ ] Verification screenshot captured and attached above.

Proceed to [Module 5.1: Raspberry Pi Workload Strategy & Pi-hole v6 DNS Deployment](../05_pi_core_services/05_1_pi_workload_strategy_and_pihole.ipynb).